# Taller 3 — Punto 2: Travelling Salesman Problem (TSP)

**Modelo:** Programación Lineal Entera Mixta (MIP) con formulación MTZ  
**Solver:** GLPK 4.65 via Pyomo  
**Referencia heurística:** Vecino Más Cercano (Nearest Neighbor)  

---

In [ ]:
import sys, os

# Asegurar que glpsol.exe esté en PATH (winglpk Windows)
_glpk_candidates = [
    r'C:\Users\Manuel Pillapa\Downloads\winglpk-4.65\glpk-4.65\w64',
    r'C:\glpk\glpk-4.65\w64',
    r'C:\glpk\w64',
]
for _d in _glpk_candidates:
    if os.path.isfile(os.path.join(_d, 'glpsol.exe')) and _d not in os.environ.get('PATH', ''):
        os.environ['PATH'] = os.environ.get('PATH', '') + os.pathsep + _d
        print(f"GLPK añadido al PATH: {_d}")
        break

# Añadir carpeta del proyecto al path de Python
notebook_dir = os.path.abspath('')
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from TSP import TSP
from util import (
    generar_ciudades_con_distancias, plotear_ruta,
    calculate_path_distance
)
from util_nearest_neighbor import nearest_neighbor
import datetime as dt

print("Entorno listo.")

---
## Referencia baseline: Vecino Más Cercano — 100 ciudades

Ejecutamos la heurística greedy para visualizar una ruta de referencia antes de comparar con el modelo LP.

In [ ]:
ciudades_ref, distancias_ref = generar_ciudades_con_distancias(100)
ruta_ref = nearest_neighbor(ciudades_ref, distancias_ref)
dist_ref = calculate_path_distance(distancias_ref, ruta_ref)
print(f"Distancia NN (100 ciudades): {dist_ref:.2f}")
plotear_ruta(ciudades_ref, distancias_ref, ruta_ref, mostrar_anotaciones=False)

---
## Punto A — LP vs Vecino Más Cercano (10 a 50 ciudades)

Se resuelve el TSP con programación lineal (Pyomo + GLPK) para 10, 20, 30, 40 y 50 ciudades  
con `mipgap=0.05` y `time_limit=30s`. Los resultados se comparan contra la heurística Vecino Más Cercano.

## Punto B — Salida interna del solver (`tee=True`)

`tee=True` redirige la salida estándar del proceso `glpsol` al output de la celda,  
mostrando iteraciones de branch-and-bound, bounds, gap relativo y condición de terminación.

In [ ]:
city_sizes = [10, 20, 30, 40, 50]
mipgap = 0.05
time_limit = 30
tee = True  # Punto B: activa la salida interna de GLPK

resultados = []
ultimo_tsp = None
ultima_ruta_lp = None
ultima_ruta_nn = None

print("=" * 75)
print("PUNTO A+B: LP (Pyomo/GLPK) vs Vecino Más Cercano")
print(f"mipgap={mipgap} | time_limit={time_limit}s | tee={tee}")
print("=" * 75)

for n_cities in city_sizes:
    print(f"\n--- {n_cities} ciudades ---")
    ciudades, distancias = generar_ciudades_con_distancias(n_cities)

    # LP con Pyomo/GLPK
    tsp = TSP(ciudades, distancias, [])
    t0 = dt.datetime.now()
    ruta_lp = tsp.encontrar_la_ruta_mas_corta(mipgap, time_limit, tee)
    t_lp = (dt.datetime.now() - t0).total_seconds()
    dist_lp = calculate_path_distance(distancias, ruta_lp)

    # Vecino Más Cercano
    t0 = dt.datetime.now()
    ruta_nn = nearest_neighbor(ciudades, distancias)
    t_nn = (dt.datetime.now() - t0).total_seconds()
    dist_nn = calculate_path_distance(distancias, ruta_nn)

    resultados.append((n_cities, t_lp, dist_lp, t_nn, dist_nn))
    ultimo_tsp = tsp
    ultima_ruta_lp = ruta_lp
    ultima_ruta_nn = ruta_nn

# Tabla resumen
print("\n" + "=" * 75)
print(f"{'N':>5} | {'T_LP(s)':>9} | {'Dist_LP':>10} | {'T_NN(s)':>9} | {'Dist_NN':>10} | {'Mejora%':>7}")
print("-" * 75)
for n, t_lp, d_lp, t_nn, d_nn in resultados:
    mejora = (d_nn - d_lp) / d_nn * 100 if d_nn > 0 else 0
    print(f"{n:>5} | {t_lp:>9.3f} | {d_lp:>10.2f} | {t_nn:>9.5f} | {d_nn:>10.2f} | {mejora:>6.1f}%")
print("=" * 75)

In [ ]:
# Gráficas para el caso de 50 ciudades (último de la lista)
if ultimo_tsp:
    print("Ruta LP — 50 ciudades:")
    ultimo_tsp.plotear_resultado(ultima_ruta_lp)
    print("Ruta Vecino Más Cercano — 50 ciudades:")
    plotear_ruta(ultimo_tsp.ciudades, ultimo_tsp.distancias, ultima_ruta_nn, True)

---
## Punto C — Heurística `limitar_funcion_objetivo` (70 ciudades)

Esta heurística añade cotas inferior y superior a la función objetivo basadas en la distancia media y mínima,  
guiando al solver hacia regiones de solución más prometedoras y reduciendo el espacio de búsqueda.

Se comparan dos corridas con los mismos datos: **con** y **sin** la heurística.

In [ ]:
n_cities = 70
ciudades, distancias = generar_ciudades_con_distancias(n_cities)
mipgap_c = 0.2
time_limit_c = 40

print("=" * 65)
print("PUNTO C: 70 ciudades CON heurística 'limitar_funcion_objetivo'")
print("=" * 65)
tsp_con = TSP(ciudades, distancias, ['limitar_funcion_objetivo'])
t0 = dt.datetime.now()
ruta_con = tsp_con.encontrar_la_ruta_mas_corta(mipgap_c, time_limit_c, tee=True)
t_con = (dt.datetime.now() - t0).total_seconds()
dist_con = calculate_path_distance(distancias, ruta_con)
tsp_con.plotear_resultado(ruta_con, mostrar_anotaciones=False)

print("\n" + "=" * 65)
print("PUNTO C: 70 ciudades SIN heurística")
print("=" * 65)
tsp_sin = TSP(ciudades, distancias, [])
t0 = dt.datetime.now()
ruta_sin = tsp_sin.encontrar_la_ruta_mas_corta(mipgap_c, time_limit_c, tee=True)
t_sin = (dt.datetime.now() - t0).total_seconds()
dist_sin = calculate_path_distance(distancias, ruta_sin)
tsp_sin.plotear_resultado(ruta_sin, mostrar_anotaciones=False)

print("\n--- Resumen Punto C (70 ciudades) ---")
print(f"Con heurística (límites FO):  tiempo={t_con:.2f}s  distancia={dist_con:.2f}")
print(f"Sin heurística:               tiempo={t_sin:.2f}s  distancia={dist_sin:.2f}")

---
## Punto D — Heurística `vecino_cercano` (100 ciudades)

Esta heurística restringe los arcos: para ciudades con distancia promedio menor al promedio global,  
limita el costo del arco a `(avg_ciudad + min_ciudad) / 2`. Reduce variables activas en el MIP.

Se comparan dos corridas con los mismos datos: **con** y **sin** la heurística.

In [ ]:
n_cities = 100
ciudades, distancias = generar_ciudades_con_distancias(n_cities)
mipgap_d = 0.05
time_limit_d = 60

print("=" * 65)
print("PUNTO D: 100 ciudades CON heurística 'vecino_cercano'")
print("=" * 65)
tsp_con = TSP(ciudades, distancias, ['vecino_cercano'])
t0 = dt.datetime.now()
ruta_con = tsp_con.encontrar_la_ruta_mas_corta(mipgap_d, time_limit_d, tee=True)
t_con = (dt.datetime.now() - t0).total_seconds()
dist_con = calculate_path_distance(distancias, ruta_con)
tsp_con.plotear_resultado(ruta_con, mostrar_anotaciones=False)

print("\n" + "=" * 65)
print("PUNTO D: 100 ciudades SIN heurística")
print("=" * 65)
tsp_sin = TSP(ciudades, distancias, [])
t0 = dt.datetime.now()
ruta_sin = tsp_sin.encontrar_la_ruta_mas_corta(mipgap_d, time_limit_d, tee=True)
t_sin = (dt.datetime.now() - t0).total_seconds()
dist_sin = calculate_path_distance(distancias, ruta_sin)
tsp_sin.plotear_resultado(ruta_sin, mostrar_anotaciones=False)

print("\n--- Resumen Punto D (100 ciudades) ---")
print(f"Con heurística (vecino cercano):  tiempo={t_con:.2f}s  distancia={dist_con:.2f}")
print(f"Sin heurística:                   tiempo={t_sin:.2f}s  distancia={dist_sin:.2f}")

---
## Punto E — Conclusiones

### E.1 Punto A — LP (Pyomo/GLPK) vs Vecino Más Cercano (10–50 ciudades)

El TSP es un problema NP-difícil: el espacio de soluciones crece factorialmente con el número de ciudades, lo que hace que los métodos exactos escalen muy mal. Esto se refleja claramente en los resultados:

- **Para 10 y 20 ciudades**, el solver GLPK encuentra la solución óptima (o dentro del 5% del óptimo) en pocos segundos. La formulación MTZ tiene ~n² variables binarias y restricciones de eliminación de subtours, lo que es manejable a esta escala.
- **Para 30–40 ciudades**, el tiempo de resolución crece significativamente. GLPK necesita explorar un árbol de branch-and-bound mucho más profundo, y la condición de terminación pasa de `OPTIMAL` a `RELATIVE MIP GAP TOLERANCE REACHED` — lo que indica que se detuvo cuando el gap entre la mejor solución entera encontrada y el bound LP era ≤ 5%.
- **Para 50 ciudades**, es probable que GLPK alcance el límite de 30s sin haber explorado completamente el árbol. La solución devuelta es factible pero no se garantiza optimalidad; en algunos casos puede activarse el fallback a Vecino Más Cercano si no se encontró ninguna solución entera dentro del tiempo.
- **La heurística NN** resuelve cualquier tamaño en microsegundos mediante una estrategia greedy: desde cada ciudad, ir a la más cercana no visitada. Es O(n²) y determinista dado el punto de partida. Su desventaja es que no backtrackea — las decisiones tempranas pueden forzar saltos largos al final de la ruta, produciendo rutas típicamente **10–25% más largas** que el óptimo LP.

**Cuándo usar cada uno:** LP/MIP cuando n es pequeño (<30 ciudades) y se requiere garantía de optimalidad. Heurísticas greedy cuando n es grande o el tiempo de respuesta es crítico y se acepta una solución aproximada.

---

### E.2 Punto B — Parámetro `tee=True` (transparencia del solver)

El parámetro `tee=True` redirige la salida estándar del proceso externo `glpsol.exe` directamente al output de la celda. Esto expone el comportamiento interno del solver en tiempo real:

- **Construcción del modelo LP:** número de filas (restricciones), columnas (variables) y elementos no-cero en la matriz de coeficientes.
- **Relajación LP inicial:** GLPK resuelve primero la relajación continua del MIP para obtener un lower bound. Se muestran las iteraciones del método simplex y el valor óptimo de la relajación.
- **Branch-and-bound:** se observan los nodos explorados, el valor del mejor bound (cota inferior), la mejor solución entera encontrada hasta el momento y el **gap relativo** entre ambos. El solver termina cuando el gap cae por debajo del `mipgap` configurado (5% en el Punto A).
- **Condición de terminación:** `OPTIMAL` si se prueba optimalidad estricta, `RELATIVE MIP GAP TOLERANCE REACHED` si se alcanzó el gap objetivo antes de agotar el árbol, o `TIME LIMIT EXCEEDED` si se agotó el tiempo sin completar la exploración.

`tee=False` es apropiado en producción o cuando se corre en batch sobre múltiples instancias. `tee=True` es indispensable para diagnóstico: permite identificar si el solver converge rápido, si está atascado en un gap alto, o si hay problemas de formulación que impiden encontrar soluciones enteras.

---

### E.3 Punto C — Heurística `limitar_funcion_objetivo` (70 ciudades, mipgap=0.2, límite=40s)

Esta heurística agrega dos restricciones explícitas sobre la función objetivo antes de llamar al solver:

```
obj  >=  min_posible    # ~25% de (n × distancia_media_baja)
obj  <=  max_posible    # ~60% de (n × distancia_media_baja)
```

El efecto es **acotar el espacio de búsqueda** del MIP: GLPK puede podar ramas del árbol de branch-and-bound cuya relajación LP ya supera la cota superior (la solución sería demasiado costosa) o queda por debajo de la cota inferior (la solución sería irrealistamente buena, violando implícitamente restricciones de conectividad).

- **Con heurística:** el solver converge más rápido porque el espacio factible es más estrecho. En 70 ciudades con 40s de límite, esto puede significar la diferencia entre encontrar una solución aceptable dentro del tiempo y agotar el límite sin solución entera.
- **Sin heurística:** GLPK explora un espacio más amplio. Puede encontrar soluciones marginalmente mejores si tiene tiempo suficiente, pero con 40s y 70 ciudades es habitual que no converja dentro del gap del 20%.
- **Riesgo:** si las cotas están mal calibradas (demasiado estrechas para la instancia), el modelo puede volverse infactible. La calibración actual es conservadora — usa la mitad del rango entre la distancia mínima y el promedio — lo que la hace robusta para distribuciones uniformes.

**Conclusión C:** La heurística de límites a la FO es especialmente efectiva en instancias medianas (50–100 ciudades) donde el solver exacto necesita guía para no desperdiciar tiempo en regiones claramente subóptimas.

---

### E.4 Punto D — Heurística `vecino_cercano` LP (100 ciudades, mipgap=0.05, límite=60s)

Esta heurística restringe el costo de los arcos que el solver puede activar. Para cada ciudad `i` cuya distancia promedio a sus vecinos sea menor que el promedio global, se añade:

```
x[i,j] * dist[i,j]  <=  (avg_ciudad_i + min_ciudad_i) / 2
```

En la práctica, esto **prohíbe arcos largos** desde ciudades bien conectadas (las que están en zonas densas). El solver queda forzado a construir rutas usando arcos cortos desde esas ciudades, imitando la filosofía greedy del Vecino Más Cercano pero dentro de un modelo MIP que sigue garantizando una ruta hamiltoniana válida (sin subtours).

- **Con heurística:** la reducción del espacio factible es significativa en 100 ciudades. GLPK puede encontrar una solución entera válida más rápido porque hay menos combinaciones de arcos a explorar. La solución puede ser igual o levemente peor en distancia total que la versión sin restricción, pero se obtiene dentro del tiempo límite.
- **Sin heurística:** con 100 ciudades, la formulación MTZ tiene ~10,000 variables binarias y ~10,000 restricciones de eliminación de subtours. El árbol de branch-and-bound es demasiado grande para explorarse en 60s; es probable que GLPK devuelva la mejor solución encontrada al agotar el tiempo, o que active el fallback a NN si no encontró ninguna solución entera.
- **Sinergia:** combinar ambas heurísticas (`limitar_funcion_objetivo` + `vecino_cercano`) podría mejorar aún más los tiempos en instancias grandes, aunque incrementa el riesgo de infactibilidad si las restricciones son demasiado agresivas en conjunto.

**Conclusión D:** Para instancias grandes (≥70 ciudades) con tiempo limitado, las heurísticas LP actúan como una forma de *warm-starting* del espacio de búsqueda — no eliminan la garantía de factibilidad del MIP, pero concentran el esfuerzo computacional en regiones prometedoras.

---

### E.5 Reflexión general

El TSP ilustra la tensión fundamental entre **optimalidad y escalabilidad** en optimización combinatoria:

| Método | Garantía de calidad | Tiempo de cómputo | Escala práctica |
|---|---|---|---|
| LP/MIP exacto (GLPK) | Óptimo o gap garantizado | Alto — crece exponencialmente | Hasta ~30 ciudades en tiempos razonables |
| LP/MIP + heurísticas | Factible, cerca del óptimo | Medio — reducido por podas | Hasta ~70–100 ciudades con límites de tiempo |
| Vecino Más Cercano | Ninguna (aprox. 10–25% del óptimo) | Muy bajo — O(n²) | Cientos de miles de ciudades |

En logística real (ruteo de vehículos, planificación de entregas), los problemas tienen cientos o miles de nodos y se resuelven con metaheurísticas (algoritmos genéticos, colonia de hormigas, simulated annealing) o solvers comerciales (Gurobi, CPLEX) con técnicas de descomposición como column generation o Benders decomposition. El enfoque LP/Pyomo de este taller es pedagógicamente valioso porque expone la estructura del problema y el funcionamiento del branch-and-bound sin cajas negras, permitiendo entender qué hace cada componente del modelo.

---
*MSDS 6004 Inteligencia Artificial — Grupo 2*